# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset using the `mlcroissant` library. All entities such as record sets, fields, and columns are referenced by their `@id` fields as defined in the Croissant schema.

### Dataset Source
The dataset schema is accessible via a Croissant-compatible URL:

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
List available record sets by `@id`, and inspect their fields and columns. This guides which `@id`s to use in subsequent code blocks.

In [ ]:
# Examine available record sets and their fields
record_sets = []
print("Record Sets in Dataset:\n-----------------------")
for record_set in metadata.record_sets:
    record_sets.append(record_set['@id'])
    print(f"@id: {record_set['@id']}")
    print(f"  Name: {record_set.get('name', 'N/A')}")
    print(f"  Description: {record_set.get('description', '')}")
    print("  Fields:")
    for field in record_set.get('field', []):
        print(f"    - @id: {field['@id']}")
        print(f"      Name: {field.get('name', 'N/A')}")
        print(f"      Data type: {field.get('dataType', 'N/A')}")
    print()

## 3. Data Extraction
Load data from a selected record set into a DataFrame using its record set `@id`.

**Example:**
For demonstration, this section will extract data from the main protein record set (replace `cr:recordSet_proteins` with the actual `@id` you want from the previous overview).

In [ ]:
# Replace with actual record set @ids from the overview above as available
main_record_set_id = record_sets[0] if len(record_sets) > 0 else None  # e.g., 'cr:recordSet_proteins'
assert main_record_set_id is not None, 'No record set found'
print(f"Loading records from record set: {main_record_set_id}\n")

dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        dataframes[record_set_id] = pd.DataFrame()

print(f"Available columns in {main_record_set_id}:\n{dataframes[main_record_set_id].columns.tolist()}")
display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We will filter, normalize, and group data for a quantitative field in the main record set. Be sure to use the relevant field `@id` (i.e., the column name from above, e.g., `'cr:field_peptide_count'` or whatever is numeric).

If unsure, run the cell above and examine the column list to pick a numeric candidate.

In [ ]:
# Identify a numeric field for EDA. Change this to match your dataset's numeric field @id
numeric_field = None
for col in dataframes[main_record_set_id].columns:
    if pd.api.types.is_numeric_dtype(dataframes[main_record_set_id][col]):
        numeric_field = col
        break
assert numeric_field is not None, "No numeric field found in main record set!"
print(f"Using numeric field: {numeric_field}")

# Filter records
threshold = dataframes[main_record_set_id][numeric_field].quantile(0.75)  # Top 25%
filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field] > threshold].copy()
print(f"Number of records with {numeric_field} above {threshold}: {len(filtered_df)}\n")
display(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field, e.g., the first non-numeric field
group_field = None
for col in dataframes[main_record_set_id].columns:
    if not pd.api.types.is_numeric_dtype(dataframes[main_record_set_id][col]):
        group_field = col
        break
if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"Mean {numeric_field} grouped by {group_field}:")
    display(grouped_df.head())

## 5. Visualization
Visualize distributions and relationships for the selected field(s).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(data=filtered_df, x=numeric_field, bins=30, kde=True)
plt.title(f"Distribution of {numeric_field} (filtered)")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# If a grouping field exists, show barplot
if group_field and group_field in filtered_df.columns:
    top_groups = filtered_df[group_field].value_counts().index[:5]
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=filtered_df[filtered_df[group_field].isin(top_groups)], x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field} (top 5 groups)")
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR^2 proteomics dataset using its Croissant schema and the `mlcroissant` library. We performed basic exploration of its record set structure, extracted a main table of protein measurements, filtered and normalized data on a quantitative field, and visualized the results. This workflow can be adapted for further downstream analysis or ML workflows.

To deepen your analysis, see the [mlcroissant documentation](https://pypi.org/project/mlcroissant/) and adjust field selection based on the Croissant schema metadata.